<a href="https://www.kaggle.com/code/mr0106/deep-past-challenge-0?scriptVersionId=301984325" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ══════════════════════════════════════════════════════════════════════════
# DEEP PAST CHALLENGE — Akkadian to English   v6.0  PRODUCTION
# ══════════════════════════════════════════════════════════════════════════
#
# ┌─────────────────────────────────────────────────────────────────────┐
# │  BEFORE SUBMITTING — ADD THE MODEL (one-time setup):               │
# │  Notebook Settings → Add Data → Models                             │
# │  Search: Helsinki-NLP/opus-mt-ar-en  → Add                        │
# │  This places it offline at /kaggle/input/opus-mt-ar-en/            │
# └─────────────────────────────────────────────────────────────────────┘
#
# WHY ALL PREVIOUS VERSIONS CRASHED:
#   Internet is DISABLED during Kaggle submission. Every previous version
#   tried to download the model at runtime → ConnectionError → Exception.
#   The ONE success (score=0.0) used a cached model but was zero-shot.
#
# ARCHITECTURE (three tiers, all offline):
#   TIER 1 — TF-IDF character-ngram retrieval   (~3 sec, always works)
#             Expected GM: ~10-18
#   TIER 2 — Fine-tuned MarianMT               (~20 min CPU, if attached)
#             Expected GM: ~20-40
#   AUTO   — Best tier wins; fallback guaranteed for every row
# ══════════════════════════════════════════════════════════════════════════

# ── CELL 1: Imports ────────────────────────────────────────────────────────
import os, re, gc, math, random, warnings, logging, time
from dataclasses import dataclass, field
from collections import defaultdict, Counter
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

import transformers
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED    = 42
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
HAS_GPU = (DEVICE == "cuda")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if HAS_GPU: torch.cuda.manual_seed_all(SEED)

print("=" * 64)
print("  DEEP PAST CHALLENGE  v6.0  —  Offline + Crash-proof")
print("=" * 64)
print(f"  Device : {DEVICE}  |  Transformers : {transformers.__version__}")
print("=" * 64)


# ── CELL 2: Config ─────────────────────────────────────────────────────────
@dataclass
class Config:
    DATA_DIR : str = ""   # auto-discovered below
    MDL_DIR  : str = "/kaggle/working/models"
    SUB_FILE : str = "/kaggle/working/submission.csv"

    # Offline model paths — scanned in order
    # User must attach "Helsinki-NLP/opus-mt-ar-en" via Kaggle Models UI
    MODEL_PATHS: List[str] = field(default_factory=lambda: [
        "/kaggle/input/opus-mt-ar-en",
        "/kaggle/input/opus-mt-ar-en/1",
        "/kaggle/input/opus-mt-ar-en/opus-mt-ar-en",
        "/kaggle/input/Helsinki-NLP-opus-mt-ar-en",
        "/kaggle/input/Helsinki-NLP-opus-mt-ar-en/1",
        "/kaggle/input/transformers-pretrained/opus-mt-ar-en",
        "/kaggle/input/marian-mt-ar-en",
        "/kaggle/input/marian-mt-ar-en/1",
        "Helsinki-NLP/opus-mt-ar-en",   # HF hub fallback (needs internet)
    ])

    # GPU
    EPOCHS_GPU : int   = 15
    LR_GPU     : float = 3e-4
    BATCH_GPU  : int   = 16
    GACC_GPU   : int   = 2
    MLEN_GPU   : int   = 256

    # CPU  (~20 min)
    EPOCHS_CPU : int   = 6
    LR_CPU     : float = 4e-4
    BATCH_CPU  : int   = 8
    GACC_CPU   : int   = 2
    MLEN_CPU   : int   = 128

    WARMUP   : float = 0.08
    WD       : float = 0.01
    LS       : float = 0.10
    CLIP     : float = 1.0
    VAL_R    : float = 0.10
    QV_N     : int   = 50
    EV_EVERY : int   = 2

    TR_BEAMS : int   = 1
    TE_BEAMS : int   = 5
    LEN_PEN  : float = 1.2
    NO_RPT   : int   = 4
    RPT_PEN  : float = 1.3

    TFIDF_K  : int   = 3        # top-k neighbours for retrieval
    TFIDF_NG : tuple = (1, 3)   # char n-gram range

    SEED   : int = SEED
    DEVICE : str = DEVICE

    def __post_init__(self):
        os.makedirs(self.MDL_DIR, exist_ok=True)
        if HAS_GPU:
            self.EPOCHS=self.EPOCHS_GPU; self.LR=self.LR_GPU
            self.BATCH=self.BATCH_GPU;   self.GACC=self.GACC_GPU
            self.MLEN=self.MLEN_GPU;     self.BINFER=32
        else:
            self.EPOCHS=self.EPOCHS_CPU; self.LR=self.LR_CPU
            self.BATCH=self.BATCH_CPU;   self.GACC=self.GACC_CPU
            self.MLEN=self.MLEN_CPU;     self.BINFER=16

CFG = Config()
print(f"  Epochs={CFG.EPOCHS}  LR={CFG.LR}  "
      f"Batch={CFG.BATCH}x{CFG.GACC}={CFG.BATCH*CFG.GACC}  MaxLen={CFG.MLEN}")


# ── CELL 3: Preprocessor ───────────────────────────────────────────────────
class Prep:
    """
    Competition Dataset Instructions preprocessing.
    Critical normalizations per the rules table:
      Ḫ/ḫ → H/h  (test uses H only; train uses Ḫ — must normalize)
      Subscripts ₂ → 2
      Remove scribal notations ! ? / < > [ ] ˹ ˺ << >>
      Gap markers: [x]→<gap>  ...→<big_gap>
      Remove determinatives {d} {ki} etc. (keep text only in transliteration)
    """
    _SUB = str.maketrans({
        '\u2080':'0','\u2081':'1','\u2082':'2','\u2083':'3','\u2084':'4',
        '\u2085':'5','\u2086':'6','\u2087':'7','\u2088':'8','\u2089':'9',
        '\u208a':'x',
    })
    _CH = str.maketrans({
        '\u1e2a':'H', '\u1e2b':'h',   # Ḫ ḫ → H h  CRITICAL
        '\u0161':'sh','\u0160':'SH',  # š Š → sh SH
        '\u1e63':'ts','\u1e62':'TS',  # ṣ Ṣ → ts TS
        '\u1e6d':'th','\u1e6c':'TH',  # ṭ Ṭ → th TH
        '\u02be':'',  '\u02bf':'',    # ʾ ʿ → (remove)
        '\u0101':'a', '\u0100':'A',   # ā Ā → a A
        '\u0113':'e', '\u0112':'E',
        '\u012b':'i', '\u012a':'I',
        '\u016b':'u', '\u016a':'U',
        '\xe1':'a2',  '\xe0':'a3',    # á à → a2 a3
        '\xe9':'e2',  '\xe8':'e3',
        '\xed':'i2',  '\xec':'i3',
        '\xfa':'u2',  '\xf9':'u3',
    })
    _DBL = re.compile(r'<<[^>]*>>')           # erroneous signs → delete
    _SCR = re.compile(r'[!?/]')              # certain/uncertain/line markers
    _HBR = re.compile(r'[\u02f9\u02fa\u2308\u2309]')  # half brackets
    _ANG = re.compile(r'[<>]')               # angle brackets (keep content)
    _GBG = re.compile(r'\[[^\]]*\.\.\.[^\]]*\]|\.\.\.|…')   # big gap
    _GSM = re.compile(r'\[\s*[xX?]\s*\]')   # small gap
    _SQB = re.compile(r'[\[\]]')             # square brackets
    _DET = re.compile(r'\{[^}]*\}')          # determinatives
    _DIV = re.compile(r'(?<!\d):(?!\d)')     # word-divider colon
    _LNO = re.compile(r'(?m)^\s*\d+\'{0,2}\s+')  # line numbers

    def src(self, t: str) -> str:
        t = self._DBL.sub(' ', t);   t = self._SCR.sub('', t)
        t = self._HBR.sub('', t);    t = self._ANG.sub('', t)
        t = self._GBG.sub(' <big_gap> ', t)
        t = self._GSM.sub(' <gap> ', t)
        t = self._SQB.sub('', t);    t = self._DET.sub('', t)
        t = self._DIV.sub(' ', t);   t = self._LNO.sub('', t)
        t = t.translate(self._SUB);  t = t.translate(self._CH)
        return re.sub(r'\s+', ' ', t).strip() or '<gap>'

    def tgt(self, t: str) -> str:
        t = re.sub(r'\s+', ' ', t).strip()
        return re.sub(r'\s+([.,;:!?])', r'\1', t)

    def __call__(self, text, mode='source') -> str:
        if text is None or (isinstance(text, float) and math.isnan(text)):
            return '<gap>'
        return self.src(str(text)) if mode == 'source' else self.tgt(str(text))

PREP = Prep()


# ── CELL 4: Metrics ─────────────────────────────────────────────────────────
class Metrics:
    """Corpus-level micro-average BLEU + chrF++ → Geometric Mean (= competition metric)."""

    @staticmethod
    def wng(s, n):
        w = s.split()
        return Counter(tuple(w[i:i+n]) for i in range(max(0, len(w)-n+1)))

    @staticmethod
    def cng(s, n):
        return Counter(s[i:i+n] for i in range(max(0, len(s)-n+1)))

    @classmethod
    def bleu(cls, P, R, mx=4):
        st = defaultdict(int)
        for p, r in zip(P, R):
            pw, rw = p.split(), r.split()
            st['h'] += len(pw); st['r'] += len(rw)
            for n in range(1, mx+1):
                pn, rn = cls.wng(p, n), cls.wng(r, n)
                st[f'm{n}'] += sum((pn & rn).values())
                st[f't{n}'] += max(len(pw)-n+1, 0)
        bp = 1.0 if st['h'] >= st['r'] else math.exp(1-st['r']/max(st['h'],1))
        lp = 0.0
        for n in range(1, mx+1):
            if not st[f't{n}'] or not st[f'm{n}']: return 0.0
            lp += math.log(st[f'm{n}']/st[f't{n}'])
        return 100.0 * bp * math.exp(lp/mx)

    @classmethod
    def chrf(cls, P, R, co=6, wo=2, beta=2.0):
        st = defaultdict(float)
        for p, r in zip(P, R):
            for n in range(1, co+1):
                pn, rn = cls.cng(p,n), cls.cng(r,n); m = sum((pn&rn).values())
                st[f'cm{n}']+=m; st[f'cp{n}']+=sum(pn.values()); st[f'cr{n}']+=sum(rn.values())
            for n in range(1, wo+1):
                pn, rn = cls.wng(p,n), cls.wng(r,n); m = sum((pn&rn).values())
                st[f'wm{n}']+=m; st[f'wp{n}']+=sum(pn.values()); st[f'wr{n}']+=sum(rn.values())
        fs = []
        for pfx, mx in [('c',co),('w',wo)]:
            for n in range(1, mx+1):
                m,p,r = st[f'{pfx}m{n}'],st[f'{pfx}p{n}'],st[f'{pfx}r{n}']
                prec = m/p if p else 0.0; rec = m/r if r else 0.0
                d = beta**2*prec+rec
                fs.append((1+beta**2)*prec*rec/d if d else 0.0)
        return 100.0*(sum(fs)/len(fs)) if fs else 0.0

    @classmethod
    def score(cls, P, R):
        b = cls.bleu(P,R); c = cls.chrf(P,R)
        return {'bleu':round(b,4),'chrf':round(c,4),
                'gm':round(math.sqrt(b*c) if b>0 and c>0 else 0.0,4)}


# ── CELL 5: TF-IDF Retrieval — TIER 1 ─────────────────────────────────────
class TFIDFTranslator:
    """
    Retrieval-based translation using character n-gram TF-IDF.
    For each test sentence, finds the most similar training source
    and returns its translation. No internet, no GPU, ~3 seconds total.
    Character n-grams are especially effective for morphologically
    complex Akkadian where word stems share many sub-sequences.
    """

    def __init__(self, train_src: List[str], train_tgt: List[str],
                 k: int = 3, ngram_range=(1, 3)):
        self.tgt = train_tgt
        self.k   = k
        self.vec = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=ngram_range,
            max_features=80000,
            sublinear_tf=True,
            min_df=1,
        )
        self.X = self.vec.fit_transform(train_src)
        print(f"  TF-IDF: {self.X.shape[0]} docs  vocab={self.X.shape[1]}")

    def translate(self, sources: List[str]) -> List[str]:
        Q    = self.vec.transform(sources)
        sims = cosine_similarity(Q, self.X)
        return [self.tgt[int(np.argmax(row))] for row in sims]

    def val_score(self, val_src, val_tgt):
        return Metrics.score(self.translate(val_src), val_tgt)


# ── CELL 6: Dataset ─────────────────────────────────────────────────────────
class Seq2SeqDS(Dataset):
    """
    Works for MarianMT.
    FIX: no as_target_tokenizer (removed in transformers >= 4.26).
    Direct tokenizer() call for labels.
    """
    def __init__(self, srcs, tgts, tok, max_len):
        self.srcs = srcs; self.tgts = tgts
        self.tok  = tok;  self.ml   = max_len

    def __len__(self): return len(self.srcs)

    def __getitem__(self, i):
        enc  = self.tok(self.srcs[i], max_length=self.ml,
                        truncation=True, padding=False)
        item = {'input_ids': enc['input_ids'],
                'attention_mask': enc['attention_mask']}
        if self.tgts is not None:
            lbl = self.tok(self.tgts[i], max_length=self.ml,
                           truncation=True, padding=False)
            item['labels'] = lbl['input_ids']
        return item


# ── CELL 7: Neural Trainer ──────────────────────────────────────────────────
class NeuralTrainer:
    def __init__(self, model, tok, cfg, name):
        self.model = model; self.tok = tok
        self.cfg = cfg; self.name = name
        self.best = -1.0
        self.ckpt = os.path.join(cfg.MDL_DIR, f'best_{name}')
        os.makedirs(self.ckpt, exist_ok=True)

    def _opt(self, lr):
        no_wd = {'bias','LayerNorm.weight','layer_norm.weight',
                 'layernorm.weight','final_layer_norm.weight'}
        return torch.optim.AdamW([
            {'params': [p for n,p in self.model.named_parameters()
                        if not any(x in n for x in no_wd)],
             'weight_decay': self.cfg.WD},
            {'params': [p for n,p in self.model.named_parameters()
                        if any(x in n for x in no_wd)],
             'weight_decay': 0.0}], lr=lr, eps=1e-8)

    @staticmethod
    def _ce(logits, labels, eps=0.1):
        V    = logits.size(-1)
        logp = torch.nn.functional.log_softmax(logits.float(), dim=-1)
        mask = labels != -100
        safe = labels.clamp(min=0)   # prevent -100 in gather (bug fix)
        nll  = -logp.gather(-1, safe.unsqueeze(-1)).squeeze(-1)
        smo  = -logp.sum(-1) / V
        return ((1-eps)*nll + eps*smo)[mask].mean()

    def _epoch(self, loader, opt, sched, ep):
        from transformers import DataCollatorForSeq2Seq
        self.model.train(); tot=0.0; n=0; opt.zero_grad()
        pad = self.tok.pad_token_id or 0
        pbar = tqdm(loader, desc=f'  Ep{ep:02d}', leave=False,
                    mininterval=30, miniters=50)
        for step, batch in enumerate(pbar):
            inp  = batch['input_ids'].to(DEVICE)
            attn = batch['attention_mask'].to(DEVICE)
            lbls = batch['labels'].to(DEVICE)
            lbls[lbls == pad] = -100
            out  = self.model(input_ids=inp, attention_mask=attn, labels=lbls)
            loss = self._ce(out.logits, lbls, self.cfg.LS)
            (loss/self.cfg.GACC).backward(); tot+=loss.item(); n+=1
            if (step+1) % self.cfg.GACC == 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.CLIP)
                opt.step(); sched.step(); opt.zero_grad()
            if n % 50 == 0: pbar.set_postfix(loss=f'{tot/n:.4f}')
        return tot / max(n, 1)

    @torch.no_grad()
    def gen(self, srcs: List[str], beams: int = 1) -> List[str]:
        """Crash-safe generation with per-sample fallback."""
        self.model.eval(); out = []
        for i in range(0, len(srcs), self.cfg.BINFER):
            batch = srcs[i:i+self.cfg.BINFER]
            try:
                enc = self.tok(batch, return_tensors='pt', padding=True,
                               truncation=True,
                               max_length=self.cfg.MLEN).to(DEVICE)
                kw = dict(max_new_tokens=self.cfg.MLEN, num_beams=beams,
                          early_stopping=(beams > 1))
                if beams > 1:
                    kw.update(length_penalty=self.cfg.LEN_PEN,
                              no_repeat_ngram_size=self.cfg.NO_RPT,
                              repetition_penalty=self.cfg.RPT_PEN)
                preds = self.model.generate(**enc, **kw)
                out.extend(self.tok.batch_decode(preds, skip_special_tokens=True))
            except Exception as e:
                print(f'    [gen warn batch {i//self.cfg.BINFER}]: {e}')
                for s in batch:
                    try:
                        enc2 = self.tok([s], return_tensors='pt', padding=True,
                                        truncation=True,
                                        max_length=self.cfg.MLEN).to(DEVICE)
                        p2 = self.model.generate(**enc2, max_new_tokens=self.cfg.MLEN,
                                                 num_beams=2, early_stopping=True)
                        out.append(self.tok.decode(p2[0], skip_special_tokens=True))
                    except:
                        out.append('')
        return out

    def fit(self, tr_ds, va_src, va_ref, lr) -> float:
        from transformers import DataCollatorForSeq2Seq, get_cosine_schedule_with_warmup
        collator = DataCollatorForSeq2Seq(self.tok, model=self.model,
                                          label_pad_token_id=-100, padding=True)
        loader = DataLoader(tr_ds, batch_size=self.cfg.BATCH, shuffle=True,
                            collate_fn=collator, num_workers=0)
        opt   = self._opt(lr)
        total = (len(loader)//self.cfg.GACC) * self.cfg.EPOCHS
        warm  = max(1, int(total * self.cfg.WARMUP))
        sched = get_cosine_schedule_with_warmup(opt, warm, total)
        nq = min(self.cfg.QV_N, len(va_src)); qs, qr = va_src[:nq], va_ref[:nq]

        print(f'\n  [{self.name}]  Train={len(tr_ds)} Val(quick)={nq} '
              f'Steps={total} LR={lr}')

        for ep in range(1, self.cfg.EPOCHS+1):
            t0   = time.time()
            loss = self._epoch(loader, opt, sched, ep)
            gp   = self.gen(qs, beams=1)
            gm_q = Metrics.score(gp, qr)
            dt   = time.time() - t0
            flag = ''
            if ep % self.cfg.EV_EVERY == 0 or ep == self.cfg.EPOCHS:
                fp = self.gen(va_src, beams=self.cfg.TR_BEAMS)
                fm = Metrics.score(fp, va_ref)
                if fm['gm'] > self.best:
                    self.best = fm['gm']; flag = '  BEST'
                    self.model.save_pretrained(self.ckpt)
                    self.tok.save_pretrained(self.ckpt)
                print(f'  Ep{ep:02d} loss={loss:.4f} quick={gm_q["gm"]:.3f} '
                      f'val={fm["gm"]:.4f} {dt:.0f}s{flag}')
            else:
                print(f'  Ep{ep:02d} loss={loss:.4f} quick={gm_q["gm"]:.3f} {dt:.0f}s')

        print(f'  Best [{self.name}]: {self.best:.4f}')
        return self.best


# ── CELL 8: Load Data ───────────────────────────────────────────────────────
print('\n' + '='*64 + '\n  LOADING DATA\n' + '='*64)

# ── Auto-discover data directory (handles any Kaggle path) ─────────────────
import glob as _glob

def _find_data_dir() -> str:
    """Scan /kaggle/input for the directory containing train.csv."""
    # Try exact known paths first
    known = [
        '/kaggle/input/deep-past-initiative-machine-translation',
        '/kaggle/input/deep-past-challenge',
        '/kaggle/input/deep-past-translate-akkadian',
        '/kaggle/input/competitions/deep-past-initiative-machine-translation',
        '/kaggle/input/akkadian-translation',
    ]
    for d in known:
        if os.path.exists(os.path.join(d, 'train.csv')):
            return d
    # Glob scan: find any train.csv under /kaggle/input/
    hits = _glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    # Last resort: current working directory
    for d in ['/kaggle/working', '.']:
        if os.path.exists(os.path.join(d, 'train.csv')):
            return d
    raise FileNotFoundError(
        "Cannot find train.csv. Attach the competition dataset via "
        "Notebook Settings → Add Data → Competition Data.")

CFG.DATA_DIR = _find_data_dir()
print(f'  DATA_DIR = {CFG.DATA_DIR}')

train_df = pd.read_csv(f'{CFG.DATA_DIR}/train.csv')
test_df  = pd.read_csv(f'{CFG.DATA_DIR}/test.csv')
print(f'  Train: {len(train_df)} rows  cols={list(train_df.columns)}')
print(f'  Test:  {len(test_df)} rows   cols={list(test_df.columns)}')

def fc(df, opts):
    for c in opts:
        if c in df.columns: return c
    for c in df.columns:
        for o in opts:
            if o.lower() in c.lower(): return c
    return df.columns[-1]

SC = fc(train_df, ['transliteration','source','akk','text'])
TC = fc(train_df, ['translation','target','en','english'])
ES = fc(test_df,  ['transliteration','source','akk','text'])
EI = fc(test_df,  ['id','index','oare_id'])

train_df['src'] = train_df[SC].apply(lambda x: PREP(x, 'source'))
train_df['tgt'] = train_df[TC].apply(lambda x: PREP(x, 'target'))
test_df['src']  = test_df[ES].apply(lambda x: PREP(x, 'source'))

mask = (train_df['src'].str.len() > 2) & (train_df['tgt'].str.len() > 2)
train_df = train_df[mask].reset_index(drop=True)
print(f'  Clean pairs: {len(train_df)}')

tr, va = train_test_split(train_df, test_size=CFG.VAL_R, random_state=SEED)
tr = tr.reset_index(drop=True); va = va.reset_index(drop=True)

ALL_SRC  = train_df['src'].tolist(); ALL_TGT  = train_df['tgt'].tolist()
TR_SRC   = tr['src'].tolist();       TR_TGT   = tr['tgt'].tolist()
VA_SRC   = va['src'].tolist();       VA_TGT   = va['tgt'].tolist()
TEST_SRC = test_df['src'].tolist()

try:    TEST_IDS = test_df[EI].astype(int).tolist()
except: TEST_IDS = test_df[EI].tolist()

print(f'  TR={len(TR_SRC)} VA={len(VA_SRC)} ALL={len(ALL_SRC)} TEST={len(TEST_SRC)}')
print(f'  SRC: {TR_SRC[0][:80]}')
print(f'  TGT: {TR_TGT[0][:80]}')


# ── CELL 9: TIER 1 — TF-IDF Retrieval ──────────────────────────────────────
print('\n' + '='*64)
print('  TIER 1: TF-IDF Retrieval  (offline, guaranteed)')
print('='*64)

retriever    = TFIDFTranslator(ALL_SRC, ALL_TGT, CFG.TFIDF_K, CFG.TFIDF_NG)
tfidf_preds  = retriever.translate(TEST_SRC)
tv           = retriever.val_score(VA_SRC, VA_TGT)
TFIDF_SCORE  = tv['gm']
print(f'  TF-IDF val: BLEU={tv["bleu"]:.2f} chrF={tv["chrf"]:.2f} GM={TFIDF_SCORE:.4f}')


# ── CELL 10: TIER 2 — Fine-tuned MarianMT (offline model required) ─────────
print('\n' + '='*64)
print('  TIER 2: Fine-tuned MarianMT  (offline, model must be attached)')
print('  → Settings > Add Data > Models > Helsinki-NLP/opus-mt-ar-en')
print('='*64)

def find_model_path(candidates: List[str]) -> Optional[str]:
    """Scan candidate paths for a valid MarianMT model directory."""
    for path in candidates:
        try:
            if os.path.exists(os.path.join(path, 'config.json')):
                return path
            if os.path.exists(os.path.join(path, 'source.spm')):
                return path
            if os.path.isdir(path):
                for sub in sorted(os.listdir(path)):
                    sp = os.path.join(path, sub)
                    if os.path.isdir(sp) and os.path.exists(
                            os.path.join(sp, 'config.json')):
                        return sp
        except:
            pass
    return None

from transformers import MarianTokenizer, MarianMTModel

NEURAL_OK    = False
NEURAL_SCORE = 0.0
neural_preds : List[str] = []
m_tok = None
m_mdl = None

model_path = find_model_path(CFG.MODEL_PATHS)
print(f'  Model path: {model_path or "NOT FOUND"}')

if model_path is not None:
    try:
        m_tok = MarianTokenizer.from_pretrained(model_path)
        m_mdl = MarianMTModel.from_pretrained(model_path).to(DEVICE)
        npar  = sum(p.numel() for p in m_mdl.parameters())
        print(f'  Loaded: {npar/1e6:.0f}M params')

        # ── Phase 1: fine-tune on 90% of data ─────────────────────────────
        print('\n  Phase 1: fine-tune on train split ...')
        tr_ds    = Seq2SeqDS(TR_SRC, TR_TGT, m_tok, CFG.MLEN)
        trainer  = NeuralTrainer(m_mdl, m_tok, CFG, 'phase1')
        NEURAL_SCORE = trainer.fit(tr_ds, VA_SRC, VA_TGT, lr=CFG.LR)
        m_mdl = MarianMTModel.from_pretrained(trainer.ckpt).to(DEVICE)
        gc.collect()

        # ── Phase 2: retrain on ALL data with lower LR ────────────────────
        ep_p2 = max(2, CFG.EPOCHS // 2)
        print(f'\n  Phase 2: retrain on ALL {len(ALL_SRC)} pairs ({ep_p2} ep) ...')
        all_ds   = Seq2SeqDS(ALL_SRC, ALL_TGT, m_tok, CFG.MLEN)
        trainer2 = NeuralTrainer(m_mdl, m_tok, CFG, 'phase2')
        trainer2.cfg.EPOCHS = ep_p2
        s2 = trainer2.fit(all_ds, VA_SRC, VA_TGT, lr=CFG.LR * 0.5)
        if s2 > NEURAL_SCORE:
            NEURAL_SCORE = s2
            m_mdl = MarianMTModel.from_pretrained(trainer2.ckpt).to(DEVICE)
        m_mdl.eval(); gc.collect()

        # ── Generate neural test predictions ──────────────────────────────
        print(f'\n  Generating {len(TEST_SRC)} test predictions (beam={CFG.TE_BEAMS}) ...')
        inf = NeuralTrainer(m_mdl, m_tok, CFG, 'inference')
        neural_preds = inf.gen(TEST_SRC, beams=CFG.TE_BEAMS)
        NEURAL_OK = True
        print(f'  Neural predictions: {len(neural_preds)}')

    except Exception as e:
        print(f'  Neural tier failed: {e}')
        print('  Falling back to TF-IDF.')

if HAS_GPU: torch.cuda.empty_cache()
gc.collect()


# ── CELL 11: Post-process + Best Tier Selection ─────────────────────────────
def clean(t: str) -> str:
    if not t or (isinstance(t, float) and math.isnan(t)):
        return 'The content of this tablet is fragmentary.'
    t = str(t).strip()
    t = re.sub(r'\s+', ' ', t)
    t = re.sub(r'\s+([.,;:!?])', r'\1', t)
    t = re.sub(r'\b(\w+)( \1\b)+', r'\1', t, flags=re.IGNORECASE)
    if t: t = t[0].upper() + t[1:]
    return t if len(t) > 3 else 'The content of this tablet section is unclear.'

print('\n' + '='*64)
print('  SELECTING BEST TIER')
print('='*64)
print(f'  TF-IDF  GM = {TFIDF_SCORE:.4f}')
print(f'  Neural  GM = {NEURAL_SCORE:.4f}  ({"ok" if NEURAL_OK else "unavailable"})')

if NEURAL_OK and NEURAL_SCORE >= TFIDF_SCORE:
    print('  → Using NEURAL predictions')
    final_preds = [clean(p) for p in neural_preds]
elif NEURAL_OK and NEURAL_SCORE < TFIDF_SCORE:
    print('  → TF-IDF scored higher; using TF-IDF predictions')
    final_preds = [clean(p) for p in tfidf_preds]
else:
    print('  → Neural unavailable; using TF-IDF predictions')
    final_preds = [clean(p) for p in tfidf_preds]

# Guarantee every row is filled
while len(final_preds) < len(TEST_SRC):
    final_preds.append('The content of this tablet is fragmentary.')
final_preds = final_preds[:len(TEST_SRC)]


# ── CELL 12: Lexicon Proper-Noun Correction ─────────────────────────────────
lex: Dict[str, str] = {}
for fn in ['OA_Lexicon_eBL.csv', 'eBL_Dictionary.csv', 'lexicon.csv']:
    fp = f'{CFG.DATA_DIR}/{fn}'
    if os.path.exists(fp):
        try:
            ld = pd.read_csv(fp, on_bad_lines='skip')
            for _, r in ld.iterrows():
                v = [str(x).strip() for x in r.tolist() if str(x).strip() != 'nan']
                if len(v) >= 2: lex[v[0].lower()] = v[1]
            print(f'  Lexicon {fn}: {len(lex)} entries')
        except Exception as e:
            print(f'  Lexicon {fn} skipped: {e}')

if lex:
    def apl(t: str) -> str:
        for k, v in lex.items():
            t = re.sub(rf'\b{re.escape(k)}\b', v, t, flags=re.IGNORECASE)
        return t
    final_preds = [apl(p) for p in final_preds]


# ── CELL 13: Save submission.csv ────────────────────────────────────────────
print('\n' + '='*64 + '\n  SAVING submission.csv\n' + '='*64)

sub = pd.DataFrame({'id': TEST_IDS, 'translation': final_preds})

try:
    sub['id'] = sub['id'].astype(int)
    sub = sub.sort_values('id')
except:
    pass

sub = sub.reset_index(drop=True)
sub['translation'] = sub['translation'].fillna(
    'The content of this tablet is fragmentary.')

sub.to_csv(CFG.SUB_FILE, index=False)

sz_kb = os.path.getsize(CFG.SUB_FILE) / 1000
print(f'\n  File   : {CFG.SUB_FILE}  ({len(sub)} rows  {sz_kb:.1f} KB)')
print(f'  Columns: {list(sub.columns)}')
print(f'  NaN    : {sub["translation"].isna().sum()}')
print(f'  Empty  : {(sub["translation"].str.len() < 4).sum()}')
print('\n  All predictions:')
print(sub.to_string())


# ── CELL 14: Final summary ──────────────────────────────────────────────────
print('\n' + '='*64 + '\n  SUMMARY\n' + '='*64)
tier_used = ('Neural' if NEURAL_OK and NEURAL_SCORE >= TFIDF_SCORE else 'TF-IDF')
print(f"""
  Tier 1  TF-IDF retrieval   GM = {TFIDF_SCORE:.4f}   (always works offline)
  Tier 2  Fine-tuned Marian  GM = {NEURAL_SCORE:.4f}   ({"OK" if NEURAL_OK else "attach model to enable"})

  Selected tier : {tier_used}
  Training data : {len(ALL_SRC)} pairs
  Test rows     : {len(sub)}   ALL FILLED

  Competition requirements met:
    File name : submission.csv  ✓
    Columns   : id, translation ✓
    No NaN    : {sub["translation"].isna().sum() == 0}  ✓
    No empty  : {(sub["translation"].str.len() < 4).sum() == 0}  ✓
    Offline   : yes ✓
""")
print('='*64)

  DEEP PAST CHALLENGE  v6.0  —  Offline + Crash-proof
  Device : cpu  |  Transformers : 5.2.0
  Epochs=6  LR=0.0004  Batch=8x2=16  MaxLen=128

  LOADING DATA
  DATA_DIR = /kaggle/input/competitions/deep-past-initiative-machine-translation
  Train: 1561 rows  cols=['oare_id', 'transliteration', 'translation']
  Test:  4 rows   cols=['id', 'text_id', 'line_start', 'line_end', 'transliteration']
  Clean pairs: 1561
  TR=1404 VA=157 ALL=1561 TEST=4
  SRC: gap ki KÙ.BABBAR a-na shi2-a-ma-tim sha-lim-a-shur i-di2-shum IGI en-na-su2-in D
  TGT: <gap> Šalim-Aššur gave him the silver for purchases. Witnessed by Enna-Suen son 

  TIER 1: TF-IDF Retrieval  (offline, guaranteed)
  TF-IDF: 1561 docs  vocab=2402
  TF-IDF val: BLEU=100.00 chrF=100.00 GM=100.0000

  TIER 2: Fine-tuned MarianMT  (offline, model must be attached)
  → Settings > Add Data > Models > Helsinki-NLP/opus-mt-ar-en
  Model path: NOT FOUND

  SELECTING BEST TIER
  TF-IDF  GM = 100.0000
  Neural  GM = 0.0000  (unavailable)
  → Ne